# Resolve reciprocal inverse fields after importing independent modules.

The two modules live in the package ``inverse_model`` beside this script.

The two modules live in the package `inverse_model` beside this notebook; the next cell writes them when they are missing, as in Colab.

In [ ]:
!pip install -q aoa-action-machine

In [ ]:
from pathlib import Path

# In Colab the package beside this notebook is not there: write it first.
package = Path("inverse_model")
if not package.exists():
    package.mkdir()
    (package / '__init__.py').write_text('"""Separate modules for the field-only inverse example."""\n')
    (package / 'domain.py').write_text('"""Share a domain without importing either relationship endpoint."""\n\nfrom aoa.action_machine.domain.base_domain import BaseDomain\n\n\nclass CrossModuleDomain(BaseDomain):\n    """Group the independently imported endpoints."""\n\n    name = "cross_module"\n    description = "Cross-module inverse resolution"\n')
    (package / 'offer.py').write_text('"""Declare the offer without importing the region at runtime."""\n\nfrom __future__ import annotations\n\nfrom typing import TYPE_CHECKING, Annotated\n\nfrom pydantic import Field\n\nfrom aoa.action_machine.domain import AssociationMany, BaseCoreEntity, Inverse\nfrom aoa.action_machine.intents.entity import entity\n\nfrom .domain import CrossModuleDomain\n\nif TYPE_CHECKING:\n    from .region import RegionCoreEntity\n\n\n@entity(description="Offer", domain=CrossModuleDomain)\nclass OfferCoreEntity(BaseCoreEntity):\n    """Collect the opposite endpoint through a forward reference."""\n\n    regions: Annotated[AssociationMany[RegionCoreEntity], Inverse(field_name="offer")] = Field(description="Regions")\n')
    (package / 'region.py').write_text('"""Declare the region without importing the offer at runtime."""\n\nfrom __future__ import annotations\n\nfrom typing import TYPE_CHECKING, Annotated\n\nfrom pydantic import Field\n\nfrom aoa.action_machine.domain import AssociationOne, BaseCoreEntity, Inverse\nfrom aoa.action_machine.intents.entity import entity\n\nfrom .domain import CrossModuleDomain\n\nif TYPE_CHECKING:\n    from .offer import OfferCoreEntity\n\n\n@entity(description="Region", domain=CrossModuleDomain)\nclass RegionCoreEntity(BaseCoreEntity):\n    """Name the reverse field without a class-valued marker."""\n\n    offer: Annotated[AssociationOne[OfferCoreEntity], Inverse(field_name="regions")] = Field(description="Offer")\n')

In [ ]:
from inverse_model.offer import OfferCoreEntity
from inverse_model.region import RegionCoreEntity

from aoa.action_machine.domain import AssociationMany
from aoa.action_machine.graph.validators.entity_field_validator import validate_entity_fields
from aoa.action_machine.graph.validators.entity_inverse_validator import validate_entity_inverses


def main() -> None:
    """Rebuild both endpoints before validating or constructing them."""
    namespace = {"OfferCoreEntity": OfferCoreEntity, "RegionCoreEntity": RegionCoreEntity}
    OfferCoreEntity.model_rebuild(_types_namespace=namespace)
    RegionCoreEntity.model_rebuild(_types_namespace=namespace)
    validate_entity_fields([OfferCoreEntity, RegionCoreEntity])
    validate_entity_inverses([OfferCoreEntity, RegionCoreEntity])
    offer = OfferCoreEntity(regions=AssociationMany(ids=("region-1",)))
    assert offer.regions.ids == ("region-1",)
    print("Cross-module reciprocal pair: accepted")

## Run

In [ ]:
main()